In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import joblib
from io import BytesIO
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

## Helper Classes

In [2]:
class AnomalyPreprocessor:
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3')
        self.df_data = None
        self.df_train = None
        self.df_valid = None
        self.df_test = None
        self.df_train_processed = None
        self.df_valid_processed = None
        self.df_test_processed = None
        self.scaler = None
        self.list_feature_cols = None

    def import_data(self):
        """Load cleaned data from S3."""
        print(f"Loading data from S3: s3://{self.str_bucket}/00_data_collection/paysim_clean.parquet")
        str_uri = f's3://{self.str_bucket}/00_data_collection/paysim_clean.parquet'
        self.df_data = pd.read_parquet(str_uri)
        print(f"Data loaded: {self.df_data.shape}")
        return self.df_data

    def feature_engineering(self):
        """Create domain-relevant features for anomaly detection."""
        if self.df_data is None:
            raise ValueError("Data not loaded. Call import_data() first.")
        
        print("\nPerforming feature engineering...")
        
        # Balance changes
        self.df_data['balance_change_orig'] = self.df_data['newbalanceOrig'] - self.df_data['oldbalanceOrg']
        self.df_data['balance_change_dest'] = self.df_data['newbalanceDest'] - self.df_data['oldbalanceDest']
        
        # Amount to balance ratio (risk indicator)
        self.df_data['amount_to_balance_ratio'] = self.df_data['amount'] / (self.df_data['oldbalanceOrg'].abs() + 1)
        
        # Balance zeroing (strong fraud signal)
        self.df_data['is_balance_zeroed'] = (self.df_data['newbalanceOrig'] == 0).astype(int)
        
        # Hour of day (temporal feature)
        self.df_data['hour_of_day'] = self.df_data['step'] % 24
        
        # One-hot encode transaction type
        df_type_encoded = pd.get_dummies(self.df_data['type'], prefix='type')
        self.df_data = pd.concat([self.df_data, df_type_encoded], axis=1)
        
        print(f"New features created. Shape now: {self.df_data.shape}")
        print(f"New columns: balance_change_orig, balance_change_dest, amount_to_balance_ratio, is_balance_zeroed, hour_of_day, {list(df_type_encoded.columns)}")
        
        return self.df_data

    def split_data(self):
        """Perform temporal train/valid/test split: 60% train (non-fraud only), 20% valid (all), 20% test (all)."""
        if self.df_data is None:
            raise ValueError("Data not loaded.")
        
        print("\nPerforming temporal train/valid/test split...")
        
        int_max_step = self.df_data['step'].max()
        int_train_threshold = int(int_max_step * 0.6)
        int_valid_threshold = int(int_max_step * 0.8)
        print(f"Train split point (step): {int_train_threshold}")
        print(f"Valid split point (step): {int_valid_threshold}")
        
        # Training: first 60% of steps, non-fraud only
        df_train_period = self.df_data[self.df_data['step'] <= int_train_threshold]
        self.df_train = df_train_period[df_train_period['isFraud'] == 0].copy()
        
        # Validation: next 20% of steps, all data
        self.df_valid = self.df_data[
            (self.df_data['step'] > int_train_threshold) & (self.df_data['step'] <= int_valid_threshold)
        ].copy()
        
        # Testing: last 20% of steps, all data
        self.df_test = self.df_data[self.df_data['step'] > int_valid_threshold].copy()
        
        print(f"Training set: {self.df_train.shape} (non-fraud only from first 60% of steps)")
        print(f"Validation set: {self.df_valid.shape} (all transactions from next 20% of steps)")
        print(f"  - Non-fraud in valid: {(self.df_valid['isFraud'] == 0).sum():,}")
        print(f"  - Fraud in valid: {(self.df_valid['isFraud'] == 1).sum():,} ({self.df_valid['isFraud'].mean()*100:.4f}%)")
        print(f"Testing set: {self.df_test.shape} (all transactions from last 20% of steps)")
        print(f"  - Non-fraud in test: {(self.df_test['isFraud'] == 0).sum():,}")
        print(f"  - Fraud in test: {(self.df_test['isFraud'] == 1).sum():,} ({self.df_test['isFraud'].mean()*100:.4f}%)")
        
        return self.df_train, self.df_valid, self.df_test

    def subsample_data(self, flt_sample_fraction=0.1):
        """Subsample training data to reduce fitting time for models."""
        if self.df_train is None:
            raise ValueError("Data not split. Call split_data() first.")

        int_original_size = len(self.df_train)
        self.df_train = self.df_train.sample(frac=flt_sample_fraction, random_state=42).reset_index(drop=True)
        int_subsampled_size = len(self.df_train)

        print(f"\nSubsampling training data ({flt_sample_fraction*100:.1f}%)...")
        print(f"Original training set size: {int_original_size:,}")
        print(f"Subsampled training set size: {int_subsampled_size:,}")

        return self.df_train

    def scale_features(self):
        """Standardize numeric features using training data statistics."""
        if self.df_train is None or self.df_valid is None or self.df_test is None:
            raise ValueError("Data not split. Call split_data() first.")
        
        print("\nScaling numeric features...")
        
        # Define features to scale (exclude identifiers and target)
        list_exclude = ['step', 'type', 'nameOrig', 'nameDest', 'isFraud', 'isFlaggedFraud']
        self.list_feature_cols = [col for col in self.df_train.columns if col not in list_exclude and self.df_train[col].dtype in [np.float64, np.int64]]
        
        print(f"Features to scale ({len(self.list_feature_cols)}): {self.list_feature_cols}")
        
        # Fit scaler on training data
        self.scaler = StandardScaler()
        self.scaler.fit(self.df_train[self.list_feature_cols])
        
        # Transform train, valid, and test
        arr_train_scaled = self.scaler.transform(self.df_train[self.list_feature_cols])
        arr_valid_scaled = self.scaler.transform(self.df_valid[self.list_feature_cols])
        arr_test_scaled = self.scaler.transform(self.df_test[self.list_feature_cols])
        
        # Create processed dataframes
        self.df_train_processed = pd.DataFrame(arr_train_scaled, columns=self.list_feature_cols, index=self.df_train.index)
        self.df_valid_processed = pd.DataFrame(arr_valid_scaled, columns=self.list_feature_cols, index=self.df_valid.index)
        self.df_test_processed = pd.DataFrame(arr_test_scaled, columns=self.list_feature_cols, index=self.df_test.index)
        
        # Keep target and identifiers
        self.df_train_processed['isFraud'] = self.df_train['isFraud'].values
        self.df_valid_processed['isFraud'] = self.df_valid['isFraud'].values
        self.df_test_processed['isFraud'] = self.df_test['isFraud'].values
        
        print(f"Scaled training set: {self.df_train_processed.shape}")
        print(f"Scaled validation set: {self.df_valid_processed.shape}")
        print(f"Scaled test set: {self.df_test_processed.shape}")
        
        return self.df_train_processed, self.df_valid_processed, self.df_test_processed

    def _upload_parquet_to_s3(self, df, str_s3_key):
        """Upload a DataFrame as parquet directly to S3."""
        buffer = BytesIO()
        df.to_parquet(buffer, index=False)
        buffer.seek(0)
        self.s3_client.put_object(Bucket=self.str_bucket, Key=str_s3_key, Body=buffer.getvalue())

    def _upload_pkl_to_s3(self, obj, str_s3_key):
        """Upload a pickled object directly to S3."""
        buffer = BytesIO()
        joblib.dump(obj, buffer)
        buffer.seek(0)
        self.s3_client.put_object(Bucket=self.str_bucket, Key=str_s3_key, Body=buffer.getvalue())

    def save_splits(self):
        """Save preprocessed splits and scaler directly to S3."""
        if self.df_train_processed is None or self.df_valid_processed is None or self.df_test_processed is None:
            raise ValueError("Data not processed. Call scale_features() first.")
        
        print("\nSaving splits to S3...")
        
        try:
            self._upload_parquet_to_s3(self.df_train_processed, '02_preprocessing/train_processed.parquet')
            self._upload_parquet_to_s3(self.df_valid_processed, '02_preprocessing/valid_processed.parquet')
            self._upload_parquet_to_s3(self.df_test_processed, '02_preprocessing/test_processed.parquet')
            self._upload_pkl_to_s3(self.scaler, '02_preprocessing/scaler.pkl')
            self._upload_pkl_to_s3(self.list_feature_cols, '02_preprocessing/feature_cols.pkl')
            print("Uploaded to S3 successfully")
        except Exception as e:
            print(f"Error uploading to S3: {e}")

## Constants

In [3]:
str_bucket = 'anomaly-detection-demo-repo'
str_task = 'preprocessing'
str_dirname_output = './output'

## Output Directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f"Output directory ready: {str_dirname_output}")

Output directory ready: ./output


## Load Data

In [5]:
preprocessor = AnomalyPreprocessor(str_bucket, str_dirname_output)
df_data = preprocessor.import_data()

Loading data from S3: s3://anomaly-detection-demo-repo/00_data_collection/paysim_clean.parquet
Data loaded: (6362620, 11)


## Feature Engineering

In [6]:
df_engineered = preprocessor.feature_engineering()


Performing feature engineering...
New features created. Shape now: (6362620, 21)
New columns: balance_change_orig, balance_change_dest, amount_to_balance_ratio, is_balance_zeroed, hour_of_day, ['type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [7]:
print("\nFeature engineering results:")
print(df_engineered[['balance_change_orig', 'balance_change_dest', 'amount_to_balance_ratio', 'is_balance_zeroed', 'hour_of_day']].describe())


Feature engineering results:
       balance_change_orig  balance_change_dest  amount_to_balance_ratio  \
count         6.362620e+06         6.362620e+06             6.362620e+06   
mean          2.123056e+04         1.242947e+05             7.067448e+04   
std           1.466433e+05         8.129391e+05             5.084243e+05   
min          -1.000000e+07        -1.306083e+07             0.000000e+00   
25%          -1.015044e+04         0.000000e+00             2.344011e-01   
50%           0.000000e+00         0.000000e+00             6.453832e+00   
75%           0.000000e+00         1.491054e+05             1.228776e+04   
max           1.915268e+06         1.056878e+08             9.244552e+07   

       is_balance_zeroed   hour_of_day  
count       6.362620e+06  6.362620e+06  
mean        5.673081e-01  1.532145e+01  
std         4.954489e-01  4.321799e+00  
min         0.000000e+00  0.000000e+00  
25%         0.000000e+00  1.200000e+01  
50%         1.000000e+00  1.600000e+01 

## Train/Valid/Test Split

In [8]:
df_train, df_valid, df_test = preprocessor.split_data()


Performing temporal train/valid/test split...
Train split point (step): 445
Valid split point (step): 594
Training set: (6005930, 21) (non-fraud only from first 60% of steps)
Validation set: (228103, 21) (all transactions from next 20% of steps)
  - Non-fraud in valid: 226,551
  - Fraud in valid: 1,552 (0.6804%)
Testing set: (123580, 21) (all transactions from last 20% of steps)
  - Non-fraud in test: 121,926
  - Fraud in test: 1,654 (1.3384%)


## Subsample Training Data

In [9]:
df_train = preprocessor.subsample_data(flt_sample_fraction=0.1)


Subsampling training data (10.0%)...
Original training set size: 6,005,930
Subsampled training set size: 600,593


## Feature Scaling

In [10]:
df_train_processed, df_valid_processed, df_test_processed = preprocessor.scale_features()


Scaling numeric features...
Features to scale (10): ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'balance_change_orig', 'balance_change_dest', 'amount_to_balance_ratio', 'is_balance_zeroed', 'hour_of_day']
Scaled training set: (600593, 11)
Scaled validation set: (228103, 11)
Scaled test set: (123580, 11)


In [11]:
print("\nScaled training set (first 5 rows):")
print(df_train_processed.head())
print(f"\nScaled training set statistics:")
print(df_train_processed.describe())


Scaled training set (first 5 rows):
     amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  newbalanceDest  \
0 -0.297934      -0.288799       -0.292855       -0.325168       -0.335254   
1  0.751625       0.376905        0.575180        0.046925       -0.162007   
2 -0.293267      -0.288785       -0.292855       -0.325168       -0.335254   
3 -0.231249       4.786116        4.731062       -0.237153       -0.265244   
4 -0.160069      -0.201943       -0.235485       -0.254364       -0.246677   

   balance_change_orig  balance_change_dest  amount_to_balance_ratio  \
0            -0.216590            -0.156589                -0.137313   
1             5.667495            -0.939404                -0.141827   
2            -0.216977            -0.156589                -0.141590   
3             0.177454            -0.209012                -0.141828   
4            -1.008224            -0.051270                -0.141827   

   is_balance_zeroed  hour_of_day  isFraud  
0           0.87

## Save Preprocessed Data

In [12]:
preprocessor.save_splits()


Saving splits to S3...
Uploaded to S3 successfully


## Completion

In [13]:
print("\n=== PREPROCESSING COMPLETE ===")
print(f"Training set: {df_train_processed.shape}")
print(f"Validation set: {df_valid_processed.shape}")
print(f"Test set: {df_test_processed.shape}")
print(f"Features: {preprocessor.list_feature_cols}")


=== PREPROCESSING COMPLETE ===
Training set: (600593, 11)
Validation set: (228103, 11)
Test set: (123580, 11)
Features: ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'balance_change_orig', 'balance_change_dest', 'amount_to_balance_ratio', 'is_balance_zeroed', 'hour_of_day']
